In [7]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
import pandas as pd

## Problem 7-3

I complete problem 3 first as a way to create functions for calculating E that I can use in problem 7-2

In [2]:
# Mass Transfer method

Ke = 1.51e-5 # m km^-1 kpa^-1 from example 7-1
Ta = 22.3    # degC
Wa = .68     # relative humidity
va = 2.16    # m/s
Ts = 23.7    # degC
P = 97.3     # kPa
Kin = 16.2   # MJ m^-2 day^-1
a = .057 
Lin = 30.6   # MJ m^-2 day^-1
ca = 1e-3    # MJ kg^-1 K^-1

# calculate saturation vapor pressure
def sat_vpr(T):
    return .611*np.exp((17.3*T)/(T + 237.3))

# calculate vapor pressure of atmosphere
def vpr_pres(EA, WA):
    return EA*WA

# convert wind sped to km/day
def m_s_to_km_day(wind):
    return wind*86400/1000

# final E calculation
def E_mass_transfer(VA, ES, EA, KE):
    return KE*VA*(ES-EA)

# surface vapor pressure 
es_star = sat_vpr(Ts)
es = es_star

# air pressure
ea_star = sat_vpr(Ta)
ea = vpr_pres(ea_star,Wa)

# wind
va_km_day = m_s_to_km_day(va)

# E
E = E_mass_transfer(va_km_day, es, ea, Ke)
print(E)

0.0031089160975449437


In [52]:
# Energy Balance Method

def K(KIN, A):
    return KIN*(1-A)

def L(LIN, A, TS):
    return LIN-.03*LIN-.97*4.9e-9*(TS+273.15)**4

# calculate lambda
def lambda_atm(T):
    return 2.5-2.36e-3*T

# calculate gamma
def gamma_atm(CA, Pres, LAMBDA):
    return (CA*Pres)/(.622*LAMBDA)

# calculate Bowen ratio
def bowen_atm(GAMMA, TS, TA, ES, EA):
    return GAMMA*((TS-TA)/(ES-EA))

# final E calculation
def E_ENERGY(k_calc, l_calc, lambda_calc, B):
    return (k_calc + l_calc)/(1000*lambda_calc*(1+B))

# calculate K
k = K(Kin, a)

# calculate L
l = L(Lin, a, Ts)

# calculate lambda
lambda_v = lambda_atm(Ts)

# calculate gamma
gamma = gamma_atm(ca, P, lambda_v)

# calculate Bowen ratio
bowen = bowen_atm(gamma, Ts, Ta, es, ea)

# calculate E
E_energy = E_ENERGY(k, l, lambda_v, bowen)

In [53]:
E_energy

0.001513409097917763

In [5]:
# Penman Equation

# calculate nabla
def nabla_atm(T_air):
    return (2508.3/(T_air + 237.3)**2)*np.exp((17.3*T_air)/(T_air + 237.3))

# Penman equation
def E_combination(NABLA,K,L,GAMMA,KE,LAMBDA,VA,EASTAR,WA):
    return (NABLA*(K+L)+GAMMA*KE*1000*LAMBDA*VA*EASTAR*(1-WA))/(1000*LAMBDA*(NABLA+GAMMA))

nabla = nabla_atm(Ta)

E_Penman = E_combination(nabla,k,l,gamma,Ke,lambda_v,va_km_day,ea_star,Wa)
print(E_Penman)

0.0030535223236151246


## Problem 7-2

In [15]:
# Upload data
df = pd.read_csv ('hw4.csv')
df.index = ['Ta', 'Wa', 'Va', 'Ts']
df = df.T
df

,Ta,Wa,Va,Ts
1,20.8,0.92,8.14,16.2
2,21.0,0.92,7.83,16.2
3,21.3,0.91,7.62,16.2
4,21.9,0.90,7.94,16.3
5,22.6,0.88,9.47,17.4
6,23.5,0.87,9.47,17.4
7,26.4,0.72,9.68,18.0
8,28.2,0.62,9.84,18.9
9,28.2,0.58,10.71,19.5
10,29.1,0.49,10.05,19.7


In [27]:
# Calculating evaporation rate for each hour and averaging results

# array of evaporation for each out
E_array_mass_transfer = []

Ke = 1.51e-5 # m km^-1 kpa^-1 from example 7-1 for Lake Hefner

# calculations for each hour
for i in range(len(df)):
    Ta = df['Ta'].iloc[i]
    Wa = df['Wa'].iloc[i]
    Va = df['Va'].iloc[i]
    Ts = df['Ts'].iloc[i]
    
    # surface vapor pressure 
    es_star = sat_vpr(Ts)
    es = es_star

    # air pressure
    ea_star = sat_vpr(Ta)
    ea = vpr_pres(ea_star,Wa)

    # wind
    va_km_day = m_s_to_km_day(Va)

    # E
    E = E_mass_transfer(va_km_day, es, ea, Ke)
    E_array_mass_transfer.append(E)

# averaging hourly results
print(np.mean(E_array_mass_transfer))

-0.001506811943744497


In [51]:
# Averaging results from table and using these results for the mass-transfer equation

Ta_avg = df[['Ta']].mean()
Wa_avg = df[['Wa']].mean()
Va_avg = df[['Va']].mean()
Ts_avg = df[['Ts']].mean()

Ta = Ta_avg.iloc[0]
Wa = Wa_avg.iloc[0]
Va = Va_avg.iloc[0]
Ts = Ts_avg.iloc[0]

# Now I calculate E with these values

# surface vapor pressure 
es_star = sat_vpr(Ts)
es = es_star

# air pressure
ea_star = sat_vpr(Ta)
ea = vpr_pres(ea_star,Wa)

# wind
va_km_day = m_s_to_km_day(Va)

# E
E = E_mass_transfer(va_km_day, es, ea, Ke)
print(E)

-0.003078516197371054
